# SemEval-2026 Task 13A — AI Code Detection Pipeline
> **Tác giả**: 25C11066  
> Mỗi cell gọi một module Python riêng biệt. Chỉ cần chỉnh `config.py`.

## Flow
```
0. Setup & Config
1. OOD Diagnosis  (diag_shift.py)
2. Feature Extraction  (feature_extractor.py)
3. Train GBM Ensemble  (train_gbm.py)
4. Fine-tune CodeBERT  (train_codebert.py)
5. Train IF+CNB  (train_ifcnb.py)
6. Soft-Voting Ensemble  (ensemble.py)
```

In [ ]:
# ── Cài đặt (chỉ cần chạy lần đầu) ─────────────────────────────────
# !pip install -r requirements.txt -q

import os, sys, logging
import numpy as np
import pandas as pd

# Thêm thư mục semeval/ vào path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')
print('✓ Imports OK')

In [ ]:
# ── Config — Chỉnh tất cả tham số ở ĐÂY ────────────────────────────
from config import CFG

cfg = CFG()

# Override nếu chạy trên Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
if IS_KAGGLE:
    cfg.train_data  = '/kaggle/input/semeval2026/train.parquet'
    cfg.val_data    = '/kaggle/input/semeval2026/validation.parquet'
    cfg.test_data   = '/kaggle/input/semeval2026/test.parquet'
    cfg.output_dir  = '/kaggle/working/outputs'
    cfg.max_train   = 100_000
    cfg.epochs      = 3

os.makedirs(cfg.output_dir, exist_ok=True)
print(f'Model: {cfg.model_name}')
print(f'Output: {cfg.output_dir}')
print(f'FP16: {cfg.fp16}')

## 1. OOD Feature Shift Diagnosis
Tính Cohen's |d| cho từng feature giữa train và test.
Features có **|d| > 1.0** là ngôn ngữ-proxy — cần loại bỏ.


In [ ]:
from diag_shift import compute_shift_report, plot_cohens_d

# ⚠️  Cần có train_features và test_features đã extract sẵn
TRAIN_FEAT = cfg.train_data.replace('.parquet', '_features.parquet')
TEST_FEAT  = cfg.test_data.replace('.parquet', '_features.parquet') if cfg.test_data else None

if TEST_FEAT and os.path.exists(TRAIN_FEAT) and os.path.exists(TEST_FEAT):
    shift_df = compute_shift_report(
        TRAIN_FEAT, TEST_FEAT,
        top_n=25,
        save_csv=os.path.join(cfg.output_dir, 'shift_report.csv')
    )
    # Vẽ biểu đồ
    plot_cohens_d(shift_df, top_n=30,
                  save_path=os.path.join(cfg.output_dir, 'cohens_d.png'))
    display(shift_df.head(20))
else:
    print('Bỏ qua: chưa có feature files. Chạy feature extraction trước.')

## 2. Feature Extraction (75+ handcrafted features)
Trích xuất song song trên train / val / test.


In [ ]:
from feature_extractor import extract_all_features
from data_utils import load_dataframe

# Định nghĩa đường dẫn feature files
TRAIN_FEAT = os.path.join(cfg.output_dir, 'train_features.parquet')
VAL_FEAT   = os.path.join(cfg.output_dir, 'val_features.parquet')
TEST_FEAT  = os.path.join(cfg.output_dir, 'test_features.parquet') if cfg.test_data else None

# ── Train ─────────────────────────────────────────────────────────────────
if not os.path.exists(TRAIN_FEAT):
    print('Extracting train features...')
    train_df = load_dataframe(cfg.train_data, cfg.max_train, 'Train', cfg.seed)
    train_feat = extract_all_features(train_df['code'], show_progress=True)
    train_feat['label'] = train_df['label'].values
    train_feat.to_parquet(TRAIN_FEAT, index=False)
    print(f'Train: {train_feat.shape} -> {TRAIN_FEAT}')
else:
    print(f'[SKIP] Train features exist: {TRAIN_FEAT}')

# ── Val ───────────────────────────────────────────────────────────────────
if not os.path.exists(VAL_FEAT):
    print('Extracting val features...')
    val_df = load_dataframe(cfg.val_data, cfg.max_val, 'Val', cfg.seed)
    val_feat = extract_all_features(val_df['code'], show_progress=True)
    val_feat['label'] = val_df['label'].values
    val_feat.to_parquet(VAL_FEAT, index=False)
    print(f'Val: {val_feat.shape} -> {VAL_FEAT}')
else:
    print(f'[SKIP] Val features exist: {VAL_FEAT}')

# ── Test (không có cột label) ─────────────────────────────────────────────
# Test set chỉ có 'code' và 'ID', không có 'label'
# Không dùng load_dataframe() vì hàm đó yêu cầu cột label
if TEST_FEAT and not os.path.exists(TEST_FEAT):
    if cfg.test_data and os.path.exists(cfg.test_data):
        print('Extracting test features...')
        test_raw = pd.read_parquet(cfg.test_data)
        test_feat = extract_all_features(test_raw['code'], show_progress=True)
        if 'ID' in test_raw.columns:
            test_feat['ID'] = test_raw['ID'].values  # giữ ID để join submission
        test_feat.to_parquet(TEST_FEAT, index=False)
        print(f'Test: {test_feat.shape} -> {TEST_FEAT}')
    else:
        print(f'WARNING: test_data not found at {cfg.test_data}')
        TEST_FEAT = None
elif TEST_FEAT and os.path.exists(TEST_FEAT):
    print(f'[SKIP] Test features exist: {TEST_FEAT}')

print(f'\nTRAIN_FEAT : {TRAIN_FEAT}')
print(f'VAL_FEAT   : {VAL_FEAT}')
print(f'TEST_FEAT  : {TEST_FEAT}')

## 3. Train Language-Robust GBM Ensemble
XGBoost + LightGBM + CatBoost với 75 features (loại 7 OOD features).


In [ ]:
# Tạm thời tắt GBM
# from train_gbm import run_gbm
# gbm_tau, val_proba_gbm = run_gbm(train_feat_path=TRAIN_FEAT, val_feat_path=VAL_FEAT, cfg=cfg, test_feat_path=TEST_FEAT)


## 4. Fine-tune CodeBERT
microsoft/codebert-base với Trainer API. Early stopping trên Macro F1.


In [ ]:
# Tạm thời tắt CodeBERT
# from train_codebert import run_codebert
# codebert_tau, val_proba_codebert = run_codebert(cfg)


## 5. Train IsolationForest + LightGBM
20 style-only features — ít bị OOD shift nhất.
LightGBM phân loại dựa trên đặc trưng style để học tương tác phi tuyến.


In [ ]:
from train_ifcnb import run_iflgbm
from data_utils import load_dataframe

train_df_full = load_dataframe(cfg.train_data, cfg.max_train, 'Train', cfg.seed)
val_df_full   = load_dataframe(cfg.val_data,   cfg.max_val,   'Val',   cfg.seed)
test_df_full  = pd.read_parquet(cfg.test_data) if cfg.test_data else None

val_proba_iflgbm = run_iflgbm(train_df_full, val_df_full, cfg, test_df=test_df_full)
print(f'val_proba_iflgbm: mean={val_proba_iflgbm.mean():.3f}')

## 6. Soft-Voting Ensemble
Rank-normalize → weighted average → quantile threshold calibration.


In [ ]:
# Tạm thời tắt Ensemble
print('Bỏ qua Ensemble để tập trung tối ưu nhánh IF+LGBM.')


## 7. Tóm tắt kết quả


In [ ]:
from sklearn.metrics import f1_score
from metrics import optimize_threshold
from data_utils import load_dataframe

val_full = load_dataframe(cfg.val_data, cfg.max_val, 'Val', cfg.seed)
y_val    = val_full['label'].values

results = {
    'IF+LGBM':   val_proba_iflgbm,
}

print('\n' + '='*55)
print(f'{"Model":<15} {"Best τ":>8} {"Val Macro F1":>14}')
print('-'*55)
for name, proba in results.items():
    tau, f1 = optimize_threshold(proba, y_val)
    print(f'{name:<15} {tau:>8.3f} {f1:>14.4f}')
print('='*55)